# 05 – Silver initial full load

The bronze `client`, `account`, `card`, `loan` tables are written to silver with type casting. Each row gets `_change_ts` = 1998-12-31 00:00:00. Change Data Feed is enabled on silver `card`.

**Code below:** adds the `src` folder to `sys.path` and imports the `silver_transforms` module.

**Output:** `src path` points to the repo's `src` folder and `tables` lists the 4 CDC tables.

In [0]:
import importlib
import os
import sys

repo_root = os.path.dirname(os.getcwd())
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

import silver_transforms
importlib.reload(silver_transforms)

print("src path:", src_path)
print("tables:", silver_transforms.CDC_TABLES)
print("initial _change_ts:", silver_transforms.INITIAL_CHANGE_TS)

## Transform preview (no write)

The 4 bronze tables are read, cast to silver types with `to_silver`, and kept in the `silver_dfs` dictionary. Nothing is written to silver.

**Output:** schema and 2 sample rows per table.

- ID columns are `integer`, date columns are `date` (`open_date`, `issued_date`, `loan_date`), `payments` is `decimal(10,2)`.

- Dates are between 1993 and 1998 (a year such as 2093 would mean the date conversion is wrong).

In [0]:
silver_dfs = {}

for table_name in silver_transforms.CDC_TABLES:
    bronze_table = silver_transforms.BRONZE_SCHEMA + "." + table_name
    df_bronze = spark.table(bronze_table)
    df_silver = silver_transforms.to_silver(df_bronze, table_name)
    silver_dfs[table_name] = df_silver

    print("=== " + table_name + " ===")
    df_silver.printSchema()
    df_silver.show(2, truncate=False)

## Silver initial full load

Each table gets `_change_ts` = 1998-12-31 00:00:00 and is written as a Delta table under `workspace.silver`.

- `overwrite`: re-running this cell returns silver to the bronze snapshot and removes the MERGE results.

- `_change_ts`: in MERGE, an update is applied only if the incoming `change_ts` is greater than this value.

**Output:** name and row count of the 4 silver tables. Expected: client 5369, account 4500, card 892, loan 682.

In [0]:
from pyspark.sql import functions as F

initial_change_ts = F.to_timestamp(F.lit(silver_transforms.INITIAL_CHANGE_TS))

for table_name in silver_transforms.CDC_TABLES:
    silver_table = silver_transforms.SILVER_SCHEMA + "." + table_name
    df_load = silver_dfs[table_name].withColumn("_change_ts", initial_change_ts)

    (
        df_load.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
    )

    row_count = spark.table(silver_table).count()
    print(silver_table, row_count)

## Change Data Feed on silver `card`

Card cancellation is a hard delete in silver, so cancelled cards are tracked with CDF (queried with `table_changes()` in 08_silver_history).

- The full load above recreates the table every time it runs, so this cell must be run again after it.

**Output:** a `delta.enableChangeDataFeed` = `true` row.

In [0]:
%sql
ALTER TABLE workspace.silver.card SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

SHOW TBLPROPERTIES workspace.silver.card;

## Check 1: NULL counts (cast errors)

`cast` and `to_date` turn values they cannot convert into NULL without an error. This cell shows the NULL count of every column in the 4 silver tables.

**Output:** one row per table, one NULL count per column. All should be 0.

In [0]:
for table_name in silver_transforms.CDC_TABLES:
    silver_table = silver_transforms.SILVER_SCHEMA + "." + table_name
    df_check = spark.table(silver_table)

    null_counts = []
    for column_name in df_check.columns:
        null_count = F.count_if(F.col(column_name).isNull()).alias(column_name)
        null_counts.append(null_count)

    print("=== " + table_name + " ===")
    df_check.select(null_counts).show(truncate=False)


## Check 2: date ranges

Checks that the `"19"` + `YYMMDD` conversion works. Berka covers 1993–1998.

**Output:** `min_date` and `max_date` for each date column. All between 1993-01-01 and 1998-12-31 (no 20xx years).

In [0]:
date_columns = {
    "account": "open_date",
    "card": "issued_date",
    "loan": "loan_date",
}

for table_name in date_columns:
    column_name = date_columns[table_name]
    silver_table = silver_transforms.SILVER_SCHEMA + "." + table_name
    df_check = spark.table(silver_table)

    df_range = df_check.select(
        F.lit(table_name).alias("table"),
        F.lit(column_name).alias("column"),
        F.min(column_name).alias("min_date"),
        F.max(column_name).alias("max_date"),
    )
    df_range.show(truncate=False)